# Results Notebook

Trains all five models, saves their results for comparison and presentation.

In [18]:
# imports
import json
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data.loader import load_dataset
from src.data.resampler import resample
from src.models.evaluation import (
	evaluate_model_predictions,
	save_classification_report,
	build_metrics_leaderboard
)
from src.visualization.data_visualization import (
	save_f1_chart,
	save_f1_comparison_chart,
	save_training_curves
)

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

### Loading and Resampling Data

In [19]:
# load the dataset

df = load_dataset()
print(f"Loaded {df.shape[0]} samples with {df.shape[1]} features.")

X = df.drop("Label", axis=1)
y = df["Label"]

Loading cleaned dataset from cache at cache/cleaned.parquet
Loaded 2522362 samples with 62 features.


In [20]:
# train test split
# 70% train, 20% test(metrics), 10% demo
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_test, X_demo, y_test, y_demo = train_test_split(X_temp, y_temp, test_size=0.333, random_state=42, stratify=y_temp)

# secondary split for models that need a validation set during training
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size = 0.1, random_state=42, stratify=y_train)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Demo set: {X_demo.shape[0]} samples")
print(f"Fit set: {X_fit.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

print(f"Total samples: {X_train.shape[0] + X_test.shape[0] + X_demo.shape[0]} samples")

Training set: 1765653 samples
Test set: 504724 samples
Demo set: 251985 samples
Fit set: 1589087 samples
Validation set: 176566 samples
Total samples: 2522362 samples


In [21]:
# save demo set for later
demo_df = X_demo.copy()
demo_df["Label"] = y_demo.values
demo_df.to_parquet(RESULTS / "demo_set.parquet", index=False)

print(f"Saved demo set to results/demo_set.parquet, with {demo_df.shape[0]} samples.")

Saved demo set to results/demo_set.parquet, with 251985 samples.


In [22]:
# resample the training set for class imbalance
X_resampled, y_resampled = resample(X_fit, y_fit)
print(f"Resampled training set: {X_resampled.shape[0]} samples")

Resampled class distribution:
Label
BENIGN                        108895
Bot                           108895
DDoS                          108895
DoS GoldenEye                 108895
DoS Hulk                      108895
DoS Slowhttptest              108895
DoS slowloris                 108895
FTP-Patator                   108895
Heartbleed                    108895
Infiltration                  108895
PortScan                      108895
SSH-Patator                   108895
Web Attack - Brute Force      108895
Web Attack - Sql Injection    108895
Web Attack - XSS              108895
Resampled training set: 1633425 samples


### Model Training

In [23]:
# logistic regression training
from src.models.logistic import train_logistic_classifier, predict_labels as lr_predict

lr_artifacts = train_logistic_classifier(
	X_resampled,
	y_resampled,
	random_state=42,
	max_iter=10,
	solver="newton-cholesky",
	verbose=1
)
print("Logistic regression training complete.")

Newton iter=1
  Check Convergence
    1. max |gradient| 0.06557669728161832 <= 0.0001 False
Newton iter=2
  Check Convergence
    1. max |gradient| 0.09067582871909617 <= 0.0001 False
Newton iter=3
  Check Convergence
    1. max |gradient| 0.0700990646864104 <= 0.0001 False
Newton iter=4
  Check Convergence
    1. max |gradient| 0.06879884268147866 <= 0.0001 False
Newton iter=5
  Check Convergence
    1. max |gradient| 0.06661075706733648 <= 0.0001 False
Newton iter=6
  Check Convergence
    1. max |gradient| 0.06250362545769422 <= 0.0001 False
Newton iter=7
  Check Convergence
    1. max |gradient| 0.05825975179276252 <= 0.0001 False
Newton iter=8
  Check Convergence
    1. max |gradient| 0.05636444493570114 <= 0.0001 False
Newton iter=9
  Check Convergence
    1. max |gradient| 0.028039650358522256 <= 0.0001 False
Newton iter=10
  Check Convergence
    1. max |gradient| 0.029057641670053497 <= 0.0001 False
Logistic regression training complete.


/Users/rbumpas/Desktop/Spring 2026/cs3540/cs3540-final/.venv/lib/python3.12/site-packages/sklearn/linear_model/_glm/_newton_solver.py:433: ConvergenceWarning: Newton solver did not converge after 10 iterations.
  warnings.warn(


In [24]:
# logistic regression evaluation
lr_result = evaluate_model_predictions("Logistic Regression", y_test, lr_predict(lr_artifacts, X_test))
save_classification_report(lr_result, RESULTS / "logistic_report.txt")

lr_report_dict = lr_result.classification_report_df.fillna(0).to_dict(orient="index")
(RESULTS / "logistic_report.json").write_text(json.dumps(lr_report_dict))

with open(RESULTS / "logistic_artifacts.pkl", "wb") as f:
    pickle.dump(lr_artifacts, f)

print(lr_result.classification_report_text)

                            precision    recall  f1-score   support

                    BENIGN      1.000     0.681     0.810    419506
                      DDoS      0.609     0.986     0.753     25616
                  DoS Hulk      0.869     0.935     0.901     34587
             DoS GoldenEye      0.364     0.948     0.526      2059
          DoS Slowhttptest      0.061     0.901     0.114      1046
                  PortScan      0.629     0.988     0.769     18173
             DoS slowloris      0.187     0.935     0.311      1077
               FTP-Patator      0.192     0.992     0.322      1187
                       Bot      0.007     0.737     0.013       391
  Web Attack - Brute Force      0.012     0.150     0.022       294
               SSH-Patator      0.125     0.910     0.220       644
          Web Attack - XSS      0.012     0.985     0.023       131
              Infiltration      0.003     0.571     0.005         7
                Heartbleed      0.024     0.500